In [1]:
!pip install -q transformers datasets accelerate scikit-learn

In [2]:
from datasets import Dataset

data = {
    "text": [
        "I forgot my password",
        "I cannot access my account",
        "I want to update my email address",

        "I was charged twice",
        "Why was my card charged?",
        "The payment amount is incorrect",

        "Where is my order?",
        "My package has not arrived",
        "When will my order be delivered?",

        "I want a refund",
        "How can I return my product?",
        "I need my money back",

        "My app keeps crashing",
        "The website is showing an error",
        "My application is not working"
    ],

    "label": [
        "account", "account", "account",
        "billing", "billing", "billing",
        "delivery", "delivery", "delivery",
        "refund", "refund", "refund",
        "technical", "technical", "technical"
    ]
}

dataset = Dataset.from_dict(data)

print(dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 15
})


In [3]:
labels = sorted(set(dataset["label"]))

label2id = {
    label: i
    for i, label in enumerate(labels)
}

id2label = {
    i: label
    for label, i in label2id.items()
}

print("Labels:", labels)
print("Label to ID:", label2id)
print("ID to Label:", id2label)

Labels: ['account', 'billing', 'delivery', 'refund', 'technical']
Label to ID: {'account': 0, 'billing': 1, 'delivery': 2, 'refund': 3, 'technical': 4}
ID to Label: {0: 'account', 1: 'billing', 2: 'delivery', 3: 'refund', 4: 'technical'}


In [4]:
def encode_labels(example):
    example["labels"] = label2id[example["label"]]
    return example

dataset = dataset.map(encode_labels)

dataset = dataset.remove_columns(["label"])

print(dataset)

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'labels'],
    num_rows: 15
})


In [5]:
dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 12
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 3
    })
})


In [6]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

print("Model and tokenizer loaded successfully!")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model and tokenizer loaded successfully!


In [7]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

print(tokenized_dataset)

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 12
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3
    })
})


In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=1,
    report_to="none"
)

print(training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start

In [9]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

def compute_metrics(eval_pred):
    logits, labels_true = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(
        labels_true,
        predictions
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels_true,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [10]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics
)

print("Trainer ready!")

Trainer ready!


In [11]:
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.648512,1.603026,0.000000,0.000000,0.000000,0.000000
2,1.554008,1.613672,0.000000,0.000000,0.000000,0.000000
3,1.538704,1.618310,0.000000,0.000000,0.000000,0.000000


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=9, training_loss=1.5948836273617215, metrics={'train_runtime': 56.1109, 'train_samples_per_second': 0.642, 'train_steps_per_second': 0.16, 'total_flos': 1192270371840.0, 'train_loss': 1.5948836273617215, 'epoch': 3.0})

In [12]:
results = trainer.evaluate()

print(results)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
1.538704,1.618310,3,0.000000,0.000000,0.000000,0.000000


{'eval_loss': 1.6183099746704102, 'eval_accuracy': 0.0, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_f1': 0.0}


In [13]:
trainer.save_model("./customer_support_model")
tokenizer.save_pretrained("./customer_support_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./customer_support_model/tokenizer_config.json',
 './customer_support_model/tokenizer.json')

In [14]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="./customer_support_model",
    tokenizer="./customer_support_model"
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [15]:
text = "I was charged twice for my order"

result = classifier(text)

print(result)

[{'label': 'technical', 'score': 0.21564321219921112}]


In [16]:
test_messages = [
    "I cannot log into my account",
    "I was charged twice for the same payment",
    "Where is my package?",
    "I want to get my money back",
    "The mobile app keeps crashing"
]

for message in test_messages:
    result = classifier(message)

    print("Message:", message)
    print("Prediction:", result[0]["label"])
    print("Score:", result[0]["score"])
    print("-" * 50)

Message: I cannot log into my account
Prediction: technical
Score: 0.2214198112487793
--------------------------------------------------
Message: I was charged twice for the same payment
Prediction: technical
Score: 0.2121984362602234
--------------------------------------------------
Message: Where is my package?
Prediction: technical
Score: 0.22177831828594208
--------------------------------------------------
Message: I want to get my money back
Prediction: technical
Score: 0.21878741681575775
--------------------------------------------------
Message: The mobile app keeps crashing
Prediction: technical
Score: 0.2347271591424942
--------------------------------------------------


In [17]:
def predict_ticket(message):
    result = classifier(message)[0]

    label = result["label"]
    score = result["score"]

    print("Customer Message:", message)
    print("Predicted Category:", label)
    print("Confidence:", round(score * 100, 2), "%")


predict_ticket("I forgot my password")

Customer Message: I forgot my password
Predicted Category: technical
Confidence: 21.67 %


In [18]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 31.2 MB/s eta 0:00:00


In [19]:
%%writefile app.py

import streamlit as st
from transformers import pipeline

# Load our trained model
classifier = pipeline(
    "text-classification",
    model="./customer_support_model",
    tokenizer="./customer_support_model"
)

# Page title
st.title("🎫 Customer Support Ticket Classifier")

st.write("Enter a customer message and the AI will predict its category.")

# Text input
message = st.text_area(
    "Customer Message",
    placeholder="Example: I was charged twice for my payment"
)

# Prediction button
if st.button("Predict"):

    if message.strip() == "":
        st.warning("Please enter a customer message.")

    else:
        result = classifier(message)[0]

        label = result["label"]
        score = result["score"]

        st.success(f"Predicted Category: {label}")
        st.info(f"Confidence: {score * 100:.2f}%")

Writing app.py


In [20]:
!pkill -f streamlit || echo "No previous Streamlit process"

^C


In [21]:
!pip install streamlit -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

!pkill streamlit || echo "No previous Streamlit process"
import time, subprocess

streamlit_proc = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(8)

# Create tunnel
!cloudflared tunnel --url http://localhost:8501 --no-autoupdate


Selecting previously unselected package cloudflared.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.10.0) ...
Setting up cloudflared (2026.10.0) ...
Processing triggers for man-db (2.12.0-4build2) ...
No previous Streamlit process
2026-10-06T10:20:53Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-06T10:20:53Z INF Requesting new quick Tunnel on trycl

In [22]:
!streamlit run app.py \
  --server.address=0.0.0.0 \
  --server.port=8501 \
  --server.headless=true



2026-10-06 10:23:22.941 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.169.129.149:8501

  Stopping...
  Stopping...
